In [5]:
# ============================================================
# QN 2
# Finite-horizon MDP for the investor problem
# Includes:
#   - exhaustive search for T = 3
#   - dynamic programming for T = 3
#   - optional DP for T = 100 (bonus)
#
# Main modelling assumptions used:
# 1) Daily change states are:
#       0 -> price unchanged
#       1 -> price increases by 10
#       2 -> price decreases by 10
# 2) Initial stock price is 100.
# 3) Initial change-state is taken to be 0 ("same"), since the question
#    gives the initial price but not the previous day's movement.
# 4) At each decision time, the investor chooses ONE action type for the day:
#       buy stock / buy futures / short / do nothing
#    and can repeat that action multiple times.
# 5) Futures bought today mature tomorrow:
#       tomorrow cash decreases by agreed price,
#       and owned stock increases by the number of futures.
# 6) If a short position is due tomorrow, then tomorrow is forced:
#       investor must buy back the shares and return them,
#       and no other transaction is allowed that day.
# 7) To keep the action set finite and reasonable:
#       - buy-stock count <= cash // price
#       - buy-futures count <= cash // price
#       - short count is capped so that worst-case next-day repurchase
#         (price+10) is coverable using current cash + short-sale proceeds,
#         after paying the 1% borrow fee today.
# ============================================================

from functools import lru_cache
from itertools import product
import time

# -------------------------
# Problem data
# -------------------------
beta = 1 / 1.05
T_small = 3
T_large = 100

P = {
    0: {0: 0.10, 1: 0.45, 2: 0.45},
    1: {0: 0.20, 1: 0.20, 2: 0.60},
    2: {0: 0.10, 1: 0.70, 2: 0.20},
}

DELTA = {
    0: 0,    # same
    1: 10,   # increase
    2: -10,  # decrease
}

SHORT_FEE_RATE = 0.01

# initial state:
# (change_state, price, cash_cents, stocks_owned, futures_held, shorts_due, bankrupt)
INITIAL_STATE = (0, 100, 10000, 1, 0, 0, 0)   # cash stored in cents


# -------------------------
# Utilities
# -------------------------
def to_dollars(cents):
    return cents / 100.0


def state_to_str(state):
    z, price, cash, stocks, futures, shorts_due, bankrupt = state
    return {
        "change_state": z,
        "price": price,
        "cash": to_dollars(cash),
        "stocks": stocks,
        "futures": futures,
        "shorts_due": shorts_due,
        "bankrupt": bool(bankrupt),
    }


def apply_next_day_price(state, next_change):
    """
    Move from end of day t to end of day t+1 BEFORE making new decision at t+1:
      - stock price moves according to next_change
      - futures from previous day mature and must be paid now
      - if shorts are due, they must be bought back now
    """
    z, price, cash, stocks, futures, shorts_due, bankrupt = state

    if bankrupt:
        return (next_change, price, cash, stocks, futures, shorts_due, 1)

    new_price = price + DELTA[next_change]

    # 1) futures mature
    if futures > 0:
        total_future_payment = futures * price * 100   # agreed price is today's price
        cash -= total_future_payment
        stocks += futures
        futures = 0
        if cash < 0:
            return (next_change, new_price, cash, stocks, futures, shorts_due, 1)

    # 2) short positions due
    if shorts_due > 0:
        repurchase_cost = shorts_due * new_price * 100
        cash -= repurchase_cost
        shorts_due = 0
        if cash < 0:
            return (next_change, new_price, cash, stocks, futures, shorts_due, 1)

    return (next_change, new_price, cash, stocks, futures, shorts_due, 0)


def enumerate_actions(state):
    """
    Return a list of feasible actions at the current end-of-day state.
    Action encoding:
        ("none", 0)
        ("buy_stock", k)
        ("buy_future", k)
        ("short", k)

    If shorts_due > 0 or bankrupt -> no choice.
    """
    z, price, cash, stocks, futures, shorts_due, bankrupt = state

    if bankrupt:
        return [("none", 0)]

    if shorts_due > 0:
        return [("none", 0)]

    actions = [("none", 0)]

    # If price is zero or negative, disallow transactions that use price in division
    # and treat the stock as non-tradeable from that point.
    if price <= 0:
        return actions

    # Buy stock now
    max_buy_stock = cash // (price * 100)
    for k in range(1, max_buy_stock + 1):
        actions.append(("buy_stock", k))

    # Buy futures now
    max_buy_future = cash // (price * 100)
    for k in range(1, max_buy_future + 1):
        actions.append(("buy_future", k))

    # Shorting only allowed if currently no stocks and no futures
    if stocks == 0 and futures == 0:
        denom = int(round((10 + SHORT_FEE_RATE * price) * 100))
        if denom > 0:
            max_short = cash // denom
            for k in range(1, max_short + 1):
                actions.append(("short", k))

    return actions


def apply_action(state, action):
    """
    Apply end-of-day action immediately at current day.
    Returns the new end-of-day state after decision.
    Stage reward is cash at hand after this action.
    """
    z, price, cash, stocks, futures, shorts_due, bankrupt = state
    a_type, k = action

    if bankrupt:
        return state

    if a_type == "none":
        return state

    if a_type == "buy_stock":
        cost = k * price * 100
        if cost > cash:
            return (z, price, cash, stocks, futures, shorts_due, 1)
        cash -= cost
        stocks += k
        return (z, price, cash, stocks, futures, shorts_due, 0)

    if a_type == "buy_future":
        # no payment today; payment due tomorrow at today's price
        # cap already enforced in enumerate_actions
        futures += k
        return (z, price, cash, stocks, futures, shorts_due, 0)

    if a_type == "short":
        # only allowed if no stocks and no futures
        if stocks != 0 or futures != 0:
            return (z, price, cash, stocks, futures, shorts_due, 1)

        fee = int(round(k * price * SHORT_FEE_RATE * 100))
        sale_proceeds = k * price * 100

        if fee > cash:
            return (z, price, cash, stocks, futures, shorts_due, 1)

        cash = cash - fee + sale_proceeds
        shorts_due += k
        return (z, price, cash, stocks, futures, shorts_due, 0)

    raise ValueError(f"Unknown action {action}")


def stage_reward(state_after_action):
    """
    r_t = cash at hand at end of day t
    """
    return state_after_action[2] / 100.0


# -------------------------
# Exhaustive search
# -------------------------
def exhaustive_value(state, t, T):
    """
    Brute-force recursion without memoization.
    Returns (best_value, best_action, policy_tree)
    """
    if t > T:
        return 0.0, None, None

    # If this state came from a short, the next day has no decision.
    # The restriction is already handled because enumerate_actions returns only "none"
    # when shorts_due > 0.
    best_val = float("-inf")
    best_action = None
    best_tree = None

    for action in enumerate_actions(state):
        s_after = apply_action(state, action)
        immediate = (beta ** t) * stage_reward(s_after)

        # If terminal stage, stop
        if t == T:
            total = immediate
            tree = {"action": action, "next": {}}
        else:
            continuation = 0.0
            subtree = {}
            for z_next, prob in P[s_after[0]].items():
                s_next = apply_next_day_price(s_after, z_next)
                val_next, a_next, tree_next = exhaustive_value(s_next, t + 1, T)
                continuation += prob * val_next
                subtree[z_next] = {
                    "prob": prob,
                    "state": state_to_str(s_next),
                    "best_action": a_next,
                    "subtree": tree_next,
                }
            total = immediate + continuation
            tree = {"action": action, "next": subtree}

        if total > best_val:
            best_val = total
            best_action = action
            best_tree = tree

    return best_val, best_action, best_tree


# -------------------------
# Dynamic programming
# -------------------------
@lru_cache(maxsize=None)
def dp_value(state, t, T):
    if t > T:
        return 0.0

    state = tuple(state)
    best_val = float("-inf")

    for action in enumerate_actions(state):
        s_after = apply_action(state, action)
        immediate = (beta ** t) * stage_reward(s_after)

        if t == T:
            total = immediate
        else:
            continuation = 0.0
            for z_next, prob in P[s_after[0]].items():
                s_next = apply_next_day_price(s_after, z_next)
                continuation += prob * dp_value(tuple(s_next), t + 1, T)
            total = immediate + continuation

        if total > best_val:
            best_val = total

    return best_val


def dp_policy(state, t, T):
    """
    Recover one optimal action at (state, t).
    """
    state = tuple(state)
    best_val = float("-inf")
    best_action = None

    for action in enumerate_actions(state):
        s_after = apply_action(state, action)
        immediate = (beta ** t) * stage_reward(s_after)

        if t == T:
            total = immediate
        else:
            continuation = 0.0
            for z_next, prob in P[s_after[0]].items():
                s_next = apply_next_day_price(s_after, z_next)
                continuation += prob * dp_value(tuple(s_next), t + 1, T)
            total = immediate + continuation

        if total > best_val:
            best_val = total
            best_action = action

    return best_action, best_val


def print_policy_tree_dp(state, t, T, indent=0, max_depth=3):
    """
    Nicely prints one optimal DP policy tree up to depth max_depth.
    """
    prefix = " " * indent
    action, val = dp_policy(state, t, T)
    print(f"{prefix}t={t}, state={state_to_str(state)}, best_action={action}, value={val:.4f}")

    if t >= T or max_depth <= 1:
        return

    s_after = apply_action(state, action)
    for z_next, prob in P[s_after[0]].items():
        s_next = apply_next_day_price(s_after, z_next)
        print(f"{prefix}  -> next change {z_next} with prob {prob}")
        print_policy_tree_dp(s_next, t + 1, T, indent + 6, max_depth - 1)

In [6]:
print("========== QN 2 : T = 3 ==========")
print("Initial state:", state_to_str(INITIAL_STATE))

start = time.perf_counter()
ex_val, ex_action, ex_tree = exhaustive_value(INITIAL_STATE, 0, T_small)
t_exhaustive = time.perf_counter() - start

start = time.perf_counter()
dp_val = dp_value(INITIAL_STATE, 0, T_small)
dp_action, dp_val_check = dp_policy(INITIAL_STATE, 0, T_small)
t_dp = time.perf_counter() - start

print(f"Exhaustive search optimal PV = {ex_val:.6f}")
print(f"Exhaustive search first action = {ex_action}")
print(f"Exhaustive search time = {t_exhaustive:.6f} seconds")

print(f"DP optimal PV = {dp_val:.6f}")
print(f"DP first action = {dp_action}")
print(f"DP time = {t_dp:.6f} seconds")

========== QN 2 : T = 3 ==========
Initial state: {'change_state': 0, 'price': 100, 'cash': 100.0, 'stocks': 1, 'futures': 0, 'shorts_due': 0, 'bankrupt': False}
Exhaustive search optimal PV = 372.324803
Exhaustive search first action = ('none', 0)
Exhaustive search time = 0.000740 seconds
DP optimal PV = 372.324803
DP first action = ('none', 0)
DP time = 0.000320 seconds


In [7]:
print("\nOne optimal DP policy tree:")
print_policy_tree_dp(INITIAL_STATE, 0, T_small, max_depth=4)


One optimal DP policy tree:
t=0, state={'change_state': 0, 'price': 100, 'cash': 100.0, 'stocks': 1, 'futures': 0, 'shorts_due': 0, 'bankrupt': False}, best_action=('none', 0), value=372.3248
  -> next change 0 with prob 0.1
      t=1, state={'change_state': 0, 'price': 100, 'cash': 100.0, 'stocks': 1, 'futures': 0, 'shorts_due': 0, 'bankrupt': False}, best_action=('none', 0), value=272.3248
        -> next change 0 with prob 0.1
            t=2, state={'change_state': 0, 'price': 100, 'cash': 100.0, 'stocks': 1, 'futures': 0, 'shorts_due': 0, 'bankrupt': False}, best_action=('none', 0), value=177.0867
              -> next change 0 with prob 0.1
                  t=3, state={'change_state': 0, 'price': 100, 'cash': 100.0, 'stocks': 1, 'futures': 0, 'shorts_due': 0, 'bankrupt': False}, best_action=('none', 0), value=86.3838
              -> next change 1 with prob 0.45
                  t=3, state={'change_state': 1, 'price': 110, 'cash': 100.0, 'stocks': 1, 'futures': 0, 'shorts_due'

In [8]:
print("\n========== QN 2 : T = 100 ==========")
dp_value.cache_clear()

start = time.perf_counter()
val_100 = dp_value(INITIAL_STATE, 0, T_large)
first_action_100, _ = dp_policy(INITIAL_STATE, 0, T_large)
t_100 = time.perf_counter() - start

print(f"DP optimal PV for T = 100 = {val_100:.6f}")
print(f"First optimal action for T = 100 = {first_action_100}")
print(f"DP time for T = 100 = {t_100:.6f} seconds")


========== QN 2 : T = 100 ==========
DP optimal PV for T = 100 = 2084.791020
First optimal action for T = 100 = ('none', 0)
DP time for T = 100 = 6.990020 seconds
